In [1]:
import ee
from tqdm import tqdm
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import geemap
from skimage.transform import resize

# import seaborn and theme it
import seaborn as sns
sns.set_theme(style="ticks", palette="pastel", font_scale=1.5, rc={"figure.figsize": (10, 7), "lines.linewidth": 2})

/Users/su28355962/Documents/codes/migration-estimation/.venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/su28355962/Documents/codes/migration-estimation/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/su28355962/Documents/codes/migration-estimation/.venv/lib/python3.9/site-packages/google/api_core/_python_version_support.py:

In [5]:
ee.Authenticate(force=True)
ee.Initialize(project="miantsa")

KeyboardInterrupt: Interrupted by user

In [ ]:
folder_path = "../data/parquet/"

path_parquet_disasters = (
    folder_path + "gd_disasters.parquet"
)

gdf_disasters = gpd.read_parquet(path_parquet_disasters)

In [ ]:
gdf_disasters.head()

In [ ]:
path_parquet_idu = (
    folder_path + "gd_idu.parquet"
)

gdf_idu = gpd.read_parquet(path_parquet_idu)

In [ ]:
gdf_idu.head()

In [ ]:
def resize_image(img_arr, target_size):
    return resize(img_arr, target_size, anti_aliasing=False)

In [ ]:
def scale_landsat(img):
    optical = img.select("SR_B.").multiply(0.0000275).add(-0.2)
    return img.addBands(optical, None, True)


def cloud_mask_landsat9(image):
    qa = image.select("QA_PIXEL")
    cloud_mask = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
    return image.updateMask(cloud_mask)

def mask_viirs_clouds(image):
    cloud_mask = image.select("QF_Cloud_Mask")
    cloud_confidence = cloud_mask.rightShift(6).bitwiseAnd(3)
    clear_mask = cloud_confidence.lte(1)
    return image.updateMask(clear_mask)

def ensure_crs_alignment(gdf, target_crs="EPSG:4326"):
    """Ensure GeoDataFrame is in the target CRS (WGS84 by default for Earth Engine)"""
    if gdf.crs != target_crs:
        print(f"Converting CRS from {gdf.crs} to {target_crs}")
        gdf = gdf.to_crs(target_crs)
    return gdf

def create_matched_geometry(point, buffer_distance, target_crs="EPSG:4326"):
    """Create Earth Engine geometry with proper CRS handling"""
    # Ensure point is in WGS84 for Earth Engine
    if hasattr(point, "to_crs"):
        point_wgs84 = point.to_crs(target_crs)
        ee_point = ee.Geometry.Point([point_wgs84.geometry.x, point_wgs84.geometry.y])
    else:
        ee_point = ee.Geometry.Point([point.geometry.x, point.geometry.y])

    buffer_geom = ee_point.buffer(buffer_distance)
    bounds_rect = buffer_geom.bounds()

    return ee_point, buffer_geom, bounds_rect


In [ ]:
def get_landsat9_rgb_median_numpy(
    point_index,
    gdf,
    buffer_distance=5000,
    start_or_end="Start date",
    scale=30,
    target_size=(330, 330),
    target_crs="EPSG:4326",
):
    gdf_aligned = ensure_crs_alignment(gdf, target_crs)

    point = gdf_aligned.iloc[point_index]
    ee_point, buffer_geom, bounds_rect = create_matched_geometry(point, buffer_distance)

    # start and end dates (30 around the event date)
    start_date = point['Start date']
    end_date = point['End date']
    if isinstance(start_date, str) or isinstance(end_date, str):
        start_date = pd.to_datetime(start_date)
        end_date = pd.to_datetime(end_date)
    date = start_date + (end_date - start_date) / 2  # Use midpoint for the date
    date_start = (date - pd.Timedelta(days=90)).strftime("%Y-%m-%d")
    date_end = (date + pd.Timedelta(days=90)).strftime("%Y-%m-%d")

    collection = (
        ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
        .filterDate(date_start, date_end)
        .filterBounds(bounds_rect)
        .filter(ee.Filter.lt("CLOUD_COVER", 60))
        .map(cloud_mask_landsat9)
        .map(scale_landsat)
    )

    collection_size = collection.size().getInfo()
    if collection_size == 0:
        print(
            f"No Landsat images found for point {point_index} between {date_start} and {date_end}"
        )
        return None, None

    composite = collection.median().clip(bounds_rect)
    rgb = composite.select(["SR_B4", "SR_B3", "SR_B2"])
    date_str = f"{date_start}_to_{date_end}"

    try:
        np_image = geemap.ee_to_numpy(
            rgb, region=bounds_rect, scale=scale
        )

        if np_image.ndim != 3:
            print(
                f"Unexpected image dimensions for point {point_index}: {np_image.shape}"
            )
            return None, None

        # Resize image
        resized = resize_image(np_image, target_size=target_size)

        return resized, date_str

    except Exception as e:
        print(f"Error processing Landsat data for point {point_index}: {str(e)}")
        return None, None

In [ ]:
# test get landsat9_rgb_median_numpy
test_index = np.random.randint(0, len(gdf_disasters))
resized_img, date_label = get_landsat9_rgb_median_numpy(
    test_index, gdf_disasters, scale=30
)
# Plotting the resized image
print(f"Resized image shape: {resized_img.shape}")
print(f"Date label: {date_label}")

# Create figure
plt.figure(figsize=(10, 8))

# Enhance the RGB visualization with contrast stretching
# Landsat typically needs enhancement for better visualization
rgb_enhanced = np.copy(resized_img)
# Apply contrast stretching - multiply by factor to enhance visibility
# but keep within valid range [0,1]
rgb_enhanced = np.clip(rgb_enhanced * 3.5, 0, 1)  

# Display the image with enhanced colors
plt.imshow(rgb_enhanced)
plt.title(f"Landsat 9 RGB on {date_label}", fontsize=14)
plt.axis("off")

# Add colorbar to show scale
plt.colorbar(label="Scaled Reflectance", shrink=0.7)
plt.tight_layout()
plt.show()

In [ ]:
def get_viirs_nightlight_median_numpy(
    point_index,
    gdf,
    buffer_distance=5000,
    start_or_end="Start date",
    scale=500,
    target_size=(20, 20),
    target_crs="EPSG:4326",
):

    # Ensure CRS alignment
    gdf_aligned = ensure_crs_alignment(gdf, target_crs)

    # Extract point and dates
    point = gdf_aligned.iloc[point_index]
    ee_point, buffer_geom, bounds_rect = create_matched_geometry(point, buffer_distance)

    date = point[start_or_end]
    if isinstance(date, str):
        date = pd.to_datetime(date)

    # two weeks around the event date
    date_start = (date - pd.Timedelta(days=7)).strftime("%Y-%m-%d")
    date_end = (date + pd.Timedelta(days=7)).strftime("%Y-%m-%d")

    # Image collection: filtered, cloud-masked
    collection = (
        ee.ImageCollection("NASA/VIIRS/002/VNP46A2")
        .filterDate(date_start, date_end)
        .filterBounds(bounds_rect)
        .select("Gap_Filled_DNB_BRDF_Corrected_NTL", "QF_Cloud_Mask", "Snow_Flag")
        .map(mask_viirs_clouds)
    )

    # Check if collection has images
    collection_size = collection.size().getInfo()
    if collection_size == 0:
        print(
            f"No VIIRS images found for point {point_index} between {date_start} and {date_end}"
        )
        return None, None

    # Median composite
    composite = collection.median().clip(bounds_rect)
    date_str = f"{date_start}_to_{date_end}"

    try:
        # Convert to NumPy array
        np_image = geemap.ee_to_numpy(
            composite.select("Gap_Filled_DNB_BRDF_Corrected_NTL"),
            region=bounds_rect,
            scale=scale,
        )

        # Handle potential 3D array
        if np_image.ndim == 3 and np_image.shape[-1] == 1:
            np_image = np_image[:, :, 0]

        # Resize image
        resized = resize_image(np_image, target_size=target_size)

        return resized, date_str

    except Exception as e:
        print(f"Error processing VIIRS data for point {point_index}: {str(e)}")
        return None, None

In [ ]:
# plot viirs nightlight
# test_index = np.random.randint(0, len(gdf_flood))
test_index = 40
resized_img_viirs, date_label_viirs = get_viirs_nightlight_median_numpy(
    test_index, gdf_disasters, buffer_distance=5000, scale=500, target_size=(20, 20)
)
# Plotting the resized image
print(f"Resized VIIRS image shape: {resized_img_viirs.shape}")
print(f"Date label: {date_label_viirs}")
# Create figure
plt.figure(figsize=(10, 8))
# Enhance the nightlight visualization with contrast stretching
# VIIRS typically needs enhancement for better visualization
viirs_enhanced = np.copy(resized_img_viirs)
# Display the image with enhanced colors
plt.imshow(viirs_enhanced)
plt.title(f"VIIRS Nightlight on {date_label_viirs}", fontsize=14)
plt.axis("off")
# Add colorbar to show scale
plt.colorbar(label="Scaled Radiance", shrink=0.7)
plt.tight_layout()
plt.show()

In [ ]:
import os
import tifffile
from PIL import Image

def download_and_save_local(
    point_indices,
    gdf,
    base_dir="disaster_dataset",
    buffer_distance=5000,
    viirs_scale=500,
    landsat_scale=30,
):
    # Create the folder structure
    folders = ["rgb", "viirs_start", "viirs_end"]
    for folder in folders:
        os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

    metadata_list = []

    for idx in tqdm(point_indices, desc="Downloading Data"):
        try:
            # 1. Fetch data using your existing functions
            # VIIRS Start
            vs_img, _ = get_viirs_nightlight_median_numpy(
                idx, gdf, buffer_distance, "Start date", viirs_scale, (20, 20)
            )
            # VIIRS End
            ve_img, _ = get_viirs_nightlight_median_numpy(
                idx, gdf, buffer_distance, "End date", viirs_scale, (20, 20)
            )
            # Landsat RGB
            rgb_img, _ = get_landsat9_rgb_median_numpy(
                idx, gdf, buffer_distance, "Start date", landsat_scale, (330, 330)
            )

            # Skip if any data is missing
            if vs_img is None or ve_img is None or rgb_img is None:
                continue

            # 2. Define Filenames
            rgb_filename = f"rgb/sample_{idx}.png"
            vs_filename = f"viirs_start/sample_{idx}.tif"
            ve_filename = f"viirs_end/sample_{idx}.tif"

            # 3. Save RGB as PNG (Scale 0-1 float to 0-255 uint8)
            rgb_uint8 = (np.clip(rgb_img, 0, 1) * 255).astype(np.uint8)
            Image.fromarray(rgb_uint8).save(os.path.join(base_dir, rgb_filename))

            # 4. Save VIIRS as 32-bit Float TIFF (Preserves scientific values)
            tifffile.imwrite(os.path.join(base_dir, vs_filename), vs_img.astype(np.float32))
            tifffile.imwrite(os.path.join(base_dir, ve_filename), ve_img.astype(np.float32))

            # 5. Track Metadata
            metadata_list.append({
                "sample_id": idx,
                "rgb_path": rgb_filename,
                "viirs_start_path": vs_filename,
                "viirs_end_path": ve_filename,
                "figures": gdf.iloc[idx]["Total figures"],
                "iso3": gdf.iloc[idx]["ISO3"],
                "type": gdf.iloc[idx]["type"]
            })

        except Exception as e:
            print(f"Error on index {idx}: {e}")

    # Save a CSV for your ML model to read
    pd.DataFrame(metadata_list).to_csv(os.path.join(base_dir, "metadata.csv"), index=False)
    print(f"\nDone! Dataset saved to: {base_dir}")

# Execute
point_indices = list(range(len(gdf_disasters)))
download_and_save_local(point_indices, gdf_disasters)